# MLPipeline Training (Colab / GPU)

Runs the same steps as the `mlpipeline_training` Airflow DAG (`dags/training_dag.py`), in-process on a Colab GPU runtime instead of as five separate Kubernetes pods.

It calls the exact same functions the DAG's pods run (`SentimentTrainer` in `src/models/training.py`, `ModelEvaluator` in `src/models/evaluation.py`, `preprocess_batch` in `src/preprocessing/text_cleaning.py`) so there is no separate logic to keep in sync. The DAG's `trigger_inference` task just kicks off the separate `mlpipeline_inference` DAG -- that's cross-DAG orchestration, not pipeline logic, so it has no equivalent here.

**Before running:** in the Colab menu, go to `Runtime > Change runtime type` and select a GPU (e.g. T4).

**Note:** Colab's local disk is wiped when the runtime disconnects. The DAG writes the trained model and metrics to a Kubernetes PersistentVolume so they survive the pod's lifetime; the equivalent here is the optional Google Drive mount in Section 1b.

## 0. Check the GPU runtime

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected -- go to Runtime > Change runtime type > GPU, then re-run this cell.")

## Clone the repo

Uses HTTPS since Colab has no SSH key configured. If the repo is private, paste a GitHub personal access token when prompted (input is hidden and not saved to the notebook); leave it blank for a public repo.

In [ ]:
import getpass
import os
import subprocess

REPO_URL = "https://github.com/rawhideron/MLPipeline.git"
BRANCH = "main"  # change if you want to run a different branch
CLONE_DIR = "/content/MLPipeline"

if not os.path.exists(CLONE_DIR):
    token = getpass.getpass("GitHub token (leave blank if the repo is public): ")
    clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "-b", BRANCH, clone_url, CLONE_DIR], check=True)
    del token, clone_url  # don't keep the token around longer than needed

os.chdir(CLONE_DIR)
print("Working directory:", os.getcwd())

## Install dependencies

Only the packages this notebook's code path actually needs (`transformers`, `datasets`, `scikit-learn`, `mlflow`, `pyyaml`), pinned to the same versions as `training/requirements.txt` -- the file that actually builds the `train_model`/`evaluate_model` pod images this notebook mirrors (`datasets` isn't in the root `requirements.txt`, but it is pinned there). `torch` is deliberately **not** reinstalled: Colab ships a GPU-matched build already, and pinning to `torch==2.12.0` here could replace it with a CPU-only or CUDA-mismatched wheel.

In [ ]:
%pip install -q transformers==5.3.0 datasets==2.16.1 scikit-learn==1.5.0 mlflow==3.11.1 pyyaml==6.0.1

In [ ]:
import logging
import sys
from pathlib import Path

import yaml
from datasets import load_dataset

sys.path.insert(0, os.getcwd())

from src.preprocessing.text_cleaning import preprocess_batch
from src.models.training import SentimentTrainer
from src.models.evaluation import ModelEvaluator

logging.basicConfig(level=logging.INFO)

CONFIG_PATH = Path("configs/training_config.yaml")

## 1. Log pipeline start

Reads `configs/training_config.yaml` and logs the model name -- same as the `log_pipeline_start` task.

In [ ]:
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

print(f"Starting training pipeline with config: {config['model']['name']}")
config

### 1b. Optional: persist output to Google Drive

The DAG's `train_model` and `evaluate_model` pods write to the `mlpipeline-serving-models` PVC, so the model and metrics survive past any single pod. Colab's local disk doesn't survive past the runtime session -- mount Drive here if you want the trained model to still be there next time.

In [ ]:
MOUNT_DRIVE = False  # set True to save the trained model to Drive instead of Colab's ephemeral disk

if MOUNT_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    config["output"]["model_path"] = "/content/drive/MyDrive/mlpipeline/trained_model"
    config["output"]["metrics_path"] = "/content/drive/MyDrive/mlpipeline/metrics.json"

print("Model will be saved to:", config["output"]["model_path"])

## 2. Validate data

Checks that the IMDB dataset is reachable from HuggingFace Hub and has the expected columns -- same check as the `validate_data` pod.

In [ ]:
print("Checking IMDB dataset accessibility...")
validation_ds = load_dataset("stanfordnlp/imdb", split="train[:100]")
assert "text" in validation_ds.features, "Missing 'text' column"
assert "label" in validation_ds.features, "Missing 'label' column"
assert len(validation_ds) == 100
print(f"Validation passed: {len(validation_ds)} examples, columns: {list(validation_ds.features)}")

## 3. Preprocess data

Runs `preprocess_batch()` -- the same function the `preprocess_data` pod calls -- on a sample batch to verify the preprocessing module is functional.

In [ ]:
print("Loading sample data for preprocessing check...")
preprocess_ds = load_dataset("stanfordnlp/imdb", split="train[:50]")
cleaned = preprocess_batch(preprocess_ds["text"], clean=True)
assert len(cleaned) == 50
print(f"Preprocessing passed: {len(cleaned)} texts cleaned")

list(zip(preprocess_ds["text"][:3], cleaned[:3]))

## 4. Train model

Runs `SentimentTrainer(config_path).train()` -- the same call the `train_model` pod makes -- to fine-tune `distilbert-base-uncased` on the full IMDB dataset per the config above and log to MLflow (local `./mlruns` unless `MLFLOW_TRACKING_URI` is set). Picks up the GPU automatically if Section 0 showed one available; on CPU this would take a very long time.

In [ ]:
RUN_TRAINING = True  # set False to skip training and just exercise steps 1-3 and 5

if RUN_TRAINING:
    trainer = SentimentTrainer(str(CONFIG_PATH))
    train_results = trainer.train()
    print(f"Training results: {train_results}")
else:
    print("RUN_TRAINING is False -- skipping. Set RUN_TRAINING = True above to train.")

## 5. Evaluate model

Runs `ModelEvaluator(model_path).evaluate()` on the IMDB test split -- the same call the `evaluate_model` pod makes -- then saves metrics and checks them against the accuracy threshold. Needs a model already saved at `config["output"]["model_path"]` (from Section 4, or a prior run if `MOUNT_DRIVE` was used).

In [ ]:
ACCURACY_THRESHOLD = 0.85

evaluator = ModelEvaluator(config["output"]["model_path"])
print(f"Model loaded from {config['output']['model_path']}")

test_dataset = load_dataset("stanfordnlp/imdb", split="test[:1000]")
metrics = evaluator.evaluate(test_dataset.iter(batch_size=32))
evaluator.save_metrics(metrics, config["output"]["metrics_path"])

accuracy = metrics["accuracy"]
print(f"Accuracy: {accuracy:.4f} (threshold: {ACCURACY_THRESHOLD})")
if accuracy < ACCURACY_THRESHOLD:
    print(f"Accuracy {accuracy:.4f} below threshold {ACCURACY_THRESHOLD} -- deployment would be blocked")

metrics

## 6. Pipeline complete

Same log line as the `log_pipeline_complete` task.

In [ ]:
print("Training pipeline completed successfully")